<a href="https://colab.research.google.com/github/Ibrah-N/PNID_Symbol_Detection_For_Large_Complex_Diagrams/blob/main/Symbol_Detection_RF_DETR_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Dataset

##Utils

In [1]:
import warnings
warnings.filterwarnings("ignore")

### PDF to Image

In [ ]:
!pip install -q PyMuPDF Pillow

In [ ]:
import fitz               # PyMuPDF
from PIL import Image, ImageFilter
import os
import shutil

def pdf_to_jpg(pdf_path, output_folder="output_images", dpi=300,
               threshold=220, output_folder_bw="bw_imgs", pdf_id=0, blur_radius=0.5):
    """
    Convert each page of a PDF to:
      • color_image_<page>.jpg
      • bw_image_<page>.jpg

    Args:
      pdf_path (str): path to .pdf file
      output_folder (str): where to save JPEGs
      dpi (int): rendering resolution
      threshold (0–255): gray cutoff for B/W
      blur_radius (float): Gaussian blur radius on B/W
    """
    os.makedirs(output_folder, exist_ok=True)
    os.makedirs(output_folder_bw, exist_ok=True)
    doc = fitz.open(pdf_path)
    page_count = doc.page_count

    for i, page in enumerate(doc, start=1):
        pix = page.get_pixmap(dpi=dpi)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        # — Color JPEG
        color_file = os.path.join(output_folder, f"color_image_{i}.jpg")
        img.save(color_file, "JPEG", quality=100, dpi=(dpi, dpi))

        # — Grayscale → threshold → blur → B/W JPEG
        gray = img.convert("L")
        bw = gray.point(lambda x: 255 if x > threshold else 0)
        bw = bw.filter(ImageFilter.GaussianBlur(blur_radius))
        bw_file = os.path.join(output_folder_bw, f"bw_image_{pdf_id}_{i}.jpg")
        bw.save(bw_file, "JPEG", quality=100, dpi=(dpi, dpi))

    doc.close()
    print(f"Done! For pdf {pdf_id} Saved {page_count} page(s) to '{output_folder}'")

if __name__ == "__main__":

  # -- single pdf parse --
  # pdf_path = "/content/90-QP20-R-001_1_Process & Inst_Diagram _PID_Legend_and_Sybmols.pdf"
  # pdf_to_jpg(pdf_path,
  #             output_folder="/content/output_images")


  # -- iterate over pdf/img files --
  input_dir = "/content/dataset_09/Equipment patches/"
  output_bw = "/content/bw_images/"
  os.makedirs(output_bw, exist_ok=True)

  # -- iterate over dir --
  for idx, file_x in enumerate(os.listdir(input_dir)):

    # -- check if pdf --
    if (file_x.endswith(".pdf") or file_x.endswith(".PDF")):
      pdf_path = os.path.join(input_dir, file_x)
      pdf_to_jpg(pdf_path,
                output_folder=output_bw,
                pdf_id = idx,
                dir_id = dirIdx,
                output_folder_bw=output_bw)

    # -- check if image --
    elif (file_x.endswith(".jpg") or file_x.endswith(".JPG")):
      src_path = os.path.join(input_dir, file_x)
      dst_path = os.path.join(output_bw, file_x)
      print(src_path, dst_path)
      shutil.copy(src_path, dst_path)
      print(f"Done! For :{file_x} Copied to {dst_path}")

    # -- else skip --
    else:
      print("Not a pdf or jpeg file -- file {}".format(file_x))

### Dataset Verification

In [2]:
###################################################
# script for verifying the dataset
# format of input data
#   - images (.jpg, .png ....)
#   - labels (yolo_format_annotations(cls, x_cen, y_cen, width, height)) # normalized values
#####################################################
import os
import cv2
import random
from google.colab.patches import cv2_imshow


def xywh_to_xyxy(x_center, y_center,
                 bbox_width, bbox_height,
                 img_width, img_height):
  """Converting normalized yolo labels (class_id, x, y, w, h) into unnormalized (class_id, x1, y1, x2, y2)

    Return:
      list (class_id, x1, y1, x2, y2)
  """

  # Denormalize YOLO coords
  x_center *= img_width
  y_center *= img_height
  bbox_width *= img_width
  bbox_height *= img_height

  x1 = int(x_center - bbox_width / 2)
  y1 = int(y_center - bbox_height / 2)
  x2 = int(x_center + bbox_width / 2)
  y2 = int(y_center + bbox_height / 2)

  return [x1, y1, x2, y2]


def draw_bbox(image, lines, line_width=2):
  """Draw bounding box over the image

    Return:
      Image array
  """
  h, w = image.shape[:2]

  for idx_line, line in enumerate(lines):

    parts = line.strip().split()
    if len(parts) != 5:
        continue

    class_id, x_center, y_center, bbox_width, bbox_height = map(float, parts)
    class_id, x1, y1, x2, y2 = int(class_id), *xywh_to_xyxy(x_center, y_center,
                                                            bbox_width, bbox_height,
                                                            w, h)

    # Pick color for this class
    color = (random.randint(0, 255),
            random.randint(0, 255),
            random.randint(0, 255))
    global colors # defined in __name__=="__main__" memory block

    if class_id not in colors.keys():

      while True:
        # prevent duplicate colors for multiple classes
        if color in colors.values():
            color = (random.randint(0, 255),
                    random.randint(0, 255),
                    random.randint(0, 255))
            continue

        else:
          # add new color for new class
          colors[class_id] = color
          break


    color = colors[class_id]

    # Draw rectangle + class ID
    cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
    cv2.putText(image, str(class_id), (x1, max(20, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

  return image


def verification_main(image_path, label_path, image_save_path):
  """Main Code looping images patha and labels path; saving resltand images

  """

  # read image
  image = cv2.imread(image_path)

  if image is None:
    print(f"Could not read image: {image_path}")
    return

  # read labels
  with open(label_path, 'r') as label_file:
    lines = label_file.readlines()

  image = draw_bbox(image, lines)

  # save image
  cv2.imwrite(image_save_path, image)
  print("Image {} Saved to {}".format(image_path, image_save_path))


### Generalizing Class Of Sub Batches

In [7]:
###################################################
# script for generalizing the class of the sub batches into global classes list
# format of input data
#   - images (.jpg, .png ....)
#   - labels (yolo_format_annotations(cls, x_cen, y_cen, width, height)) # normalized values
#
# RETURNS
# A new directory following the yolo structure
# creates a global symbols dict for combining all the classes considering the name of the class
#####################################################
import os
import cv2
import random
import shutil
import yaml
from google.colab.patches import cv2_imshow


def get_class(class_, flag='id'):
  """Extract Class Name against and ID and voice versa

  Args:
    class_ (str): Class_id if flag='id' else Class_name if flag='name'

  Return:
    class_id or class_name
  """
  global generalized_classes

  if flag == 'id': # return class_name

    if class_ not in list(generalized_classes.keys()):
      generalized_classes[len(generalized_classes)] = class_

    return generalized_classes[class_]


  elif flag == 'name': # return class_id

    if class_ not in list(generalized_classes.values()):
      generalized_classes[len(generalized_classes)] = class_

    return list(generalized_classes.keys())[list(generalized_classes.values()).index(class_)]

  else:
    raise ValueError("Flag must be 'id' or 'name'")
    return


def read_yaml_file(yaml_path):
  """Read yaml file and return the dict(Keys=class_id: Value=class_name)

    Return:
      dict
  """
  with open(yaml_path, 'r') as yaml_file:
    yaml_data = yaml.safe_load(yaml_file)

  return yaml_data['names']


def write_yaml_file(yaml_path, classes_list):
  """Write yaml file
  """
  dump_data = {

    'train': '../train/images',
    'nc': f"{len(classes_list)}",

    'names': list(classes_list.values())
  }

  with open(yaml_path, 'w') as file:
    yaml.dump(dump_data, file, sort_keys=False)

  print("Yaml file saved to {}".format(yaml_path))



def write_file(file_path, data):
  """Write data to file

  """
  with open(file_path, 'w') as file:
    for line in data:
      file.write(line + '\n')



def parse_labels(file_path, write_path, class_list):
  """Parse Yolo Data And Write Back

    Return:
  """
  # vars
  labels_ = []

  # read labels
  with open(file_path, 'r') as label_file:
    lines = label_file.readlines()

  for idx_line, line in enumerate(lines):
    parts = line.strip().split()

    if len(parts) != 5:
        continue

    class_id, x_center, y_center, bbox_width, bbox_height = map(float, parts)

    class_name = class_list[int(class_id)]

    class_id = get_class(class_name, flag='name') # return id

    labels_.append(f"{class_id} {x_center} {y_center} {bbox_width} {bbox_height}")


  # write file
  write_file(write_path, labels_)



def main(read_dir, write_dir, g_idx=0):
  """Main Code looping images patha and labels path; saving resltand images

  """

  # paths
  images_path = os.path.join(read_dir, "train", "images")
  labels_path = os.path.join(read_dir, "train", "labels")
  yaml_path = os.path.join(read_dir, "data.yaml")


  # write dataset
  images_save_path = os.path.join(write_dir, "train", "images")
  labels_save_path = os.path.join(write_dir, "train", "labels")
  yaml_path_save = os.path.join(write_dir, "data.yaml")

  os.makedirs(images_save_path, exist_ok=True)
  os.makedirs(labels_save_path, exist_ok=True)


  # read yaml file
  class_list = read_yaml_file(yaml_path)
  print(class_list)

  for img_idx, img_file in enumerate(os.listdir(images_path)):

    image_path = os.path.join(images_path, img_file)
    label_path = os.path.join(labels_path, img_file[:-3] + "txt")


    # copy image & write label txt
    save_label_path = os.path.join(labels_save_path, img_file[:-3] + "txt")
    shutil.copy(image_path, os.path.join(images_save_path, img_file))
    parse_labels(label_path, save_label_path, class_list)

    print("Image {} Copied to {} - label written to {}".format(image_path, os.path.join(images_save_path, img_file), save_label_path))


  global generalized_classes
  write_yaml_file(yaml_path_save, generalized_classes)

### Per Class Samples Analytics & Roboflow Upload

In [ ]:
###################################################
# GPT GENERATED CODE: use with precaution ...
# it extract higher, lower and middle symbols list
#####################################################

import os
import shutil
import random
import math

# =========================
# CONFIG
# =========================
INPUT_DATASET = "/content/combine_symbol_v1/train"
OUTPUT_DATASET = "/content/final_dataset"

MAX_IMAGES = 1000  # number of images to select

BETA = 0.7  # diversity weight

# Optional bucket thresholds
HIGH_THRESHOLD = 20
MEDIUM_THRESHOLD = 5

# =========================
# PATHS
# =========================
img_dir = os.path.join(INPUT_DATASET, "images")
lbl_dir = os.path.join(INPUT_DATASET, "labels")

# =========================
# FUNCTION: Extract stats
# =========================
def get_stats(label_path):
    """
    Returns:
        num_symbols (int)
        unique_classes (set)
    """
    if not os.path.exists(label_path):
        return 0, set()

    classes = set()
    count = 0

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            cls_id = int(parts[0])
            classes.add(cls_id)
            count += 1

    return count, classes

# =========================
# STEP 1: COLLECT DATA
# =========================
dataset = []

for file in os.listdir(img_dir):
    if not file.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    img_path = os.path.join(img_dir, file)
    lbl_path = os.path.join(lbl_dir, file.rsplit(".", 1)[0] + ".txt")

    num_symbols, class_set = get_stats(lbl_path)
    num_unique = len(class_set)

    # Score = density + diversity
    score = math.log(1 + num_symbols) + BETA * num_unique

    dataset.append({
        "img": img_path,
        "lbl": lbl_path,
        "score": score,
        "num_symbols": num_symbols,
        "num_unique": num_unique
    })

print(f"Total images: {len(dataset)}")

# =========================
# STEP 2: SORT BY SCORE
# =========================
dataset = sorted(dataset, key=lambda x: x["score"], reverse=True)

# =========================
# STEP 3: (OPTIONAL) BUCKETING
# =========================
high, medium, low = [], [], []

for item in dataset:
    c = item["num_symbols"]

    if c > HIGH_THRESHOLD:
        high.append(item)
    elif c > MEDIUM_THRESHOLD:
        medium.append(item)
    else:
        low.append(item)

print(f"Buckets -> High: {len(high)}, Medium: {len(medium)}, Low: {len(low)}")

# =========================
# STEP 4: SELECT IMAGES
# =========================
selected = []

def sample_from_bucket(bucket, n):
    if len(bucket) == 0:
        return []
    return bucket[:min(n, len(bucket))]  # already sorted by score

# distribute selection
n_high = int(MAX_IMAGES * 0.4)
n_medium = int(MAX_IMAGES * 0.3)
n_low = int(MAX_IMAGES * 0.3)

selected += sample_from_bucket(high, n_high)
selected += sample_from_bucket(medium, n_medium)
selected += sample_from_bucket(low, n_low)

print(f"Selected images: {len(selected)}")

# =========================
# STEP 5: COPY DATA
# =========================
out_img_dir = os.path.join(OUTPUT_DATASET, "images")
out_lbl_dir = os.path.join(OUTPUT_DATASET, "labels")

os.makedirs(out_img_dir, exist_ok=True)
os.makedirs(out_lbl_dir, exist_ok=True)

for idx, item in enumerate(selected):
    img_dst = os.path.join(out_img_dir, f"{idx}.jpg")
    lbl_dst = os.path.join(out_lbl_dir, f"{idx}.txt")

    shutil.copy(item["img"], img_dst)

    if os.path.exists(item["lbl"]):
        shutil.copy(item["lbl"], lbl_dst)

print("✅ Dataset created successfully!")

In [ ]:
from roboflow import Roboflow
from google.colab import userdata

rf = Roboflow(api_key=str(userdata.get("r_b_ibrah")))

workspace = rf.workspace("institute-of-management-sciences-eacvm")

workspace.upload_dataset(
    dataset_path="/content/final_dataset",
    project_name="symbol_dataset_fine_tune",
    num_workers=8
)

### Project Base Dataset

## Main


### Drive

In [3]:
from google.colab import drive
drive.mount("Ibrah")

Mounted at Ibrah


### Unzip Sub Batches


In [3]:
import os

zip_data_path = "/content/Ibrah/MyDrive/Symbol_Detection/Classification/Classification_Datasets/Roboflow Symbols Batch.zip"

print("Unzipping {}...".format(zip_data_path.split("/")[-1]))
!unzip -q {zip_data_path.replace(" ", "\ ")} -d /content/
print("Unzipping {} Done Sucessfully!!!".format(zip_data_path.split("/")[-1]))

Unzipping Roboflow Symbols Batch.zip...


<string>:1: SyntaxWarning: invalid escape sequence '\ '


Unzipping Roboflow Symbols Batch.zip Done Sucessfully!!!


In [4]:
import os

# -- dataset paths --
list_paths = ["/content/Roboflow Symbols Batch/First Batch.v8i.yolov8.zip", "/content/Roboflow Symbols Batch/Second_batch.v10i.yolov8.zip",
              "/content/Roboflow Symbols Batch/Third Batch Z1.v9i.yolov8.zip", "/content/Roboflow Symbols Batch/Third Batch Z2.v8i.yolov8.zip",
              "/content/Roboflow Symbols Batch/Third Batch.v8i.yolov8.zip", "/content/Roboflow Symbols Batch/batch-4-raw_annotated_dataset.v8i.yolov8.zip",
              "/content/Roboflow Symbols Batch/Batch 5 - raw_annotated_dataset.v8i.yolov8.zip", "/content/Roboflow Symbols Batch/Batch 6 - raw_annotated_dataset.v8i.yolov8.zip",
              "/content/Roboflow Symbols Batch/Batch 7- raw_annotated_dataset_05.v8i.yolov8.zip", "/content/Roboflow Symbols Batch/batch_8_raw_annotated_dataset.v7i.yolov8.zip"]

# -- save paths for each zip --
save_paths = ["dataset_01", "dataset_02", "dataset_03_01", "dataset_03_02",
              "dataset_03_03", "dataset_04", "dataset_05", "dataset_06",
              "dataset_07", "dataset_08"]


save_base_path = "/content/combine_symbol_v0"

os.makedirs(save_base_path, exist_ok=True)


for idx in range(len((list_paths))):
  print("Unziping {} to {}".format(list_paths[idx], os.path.join(save_base_path, save_paths[idx])))

  !unzip -q {list_paths[idx].replace(" ", "\ ")} -d {os.path.join(save_base_path, save_paths[idx])}

Unziping /content/Roboflow Symbols Batch/First Batch.v8i.yolov8.zip to /content/combine_symbol_v0/dataset_01


<string>:1: SyntaxWarning: invalid escape sequence '\ '


Unziping /content/Roboflow Symbols Batch/Second_batch.v10i.yolov8.zip to /content/combine_symbol_v0/dataset_02


<string>:1: SyntaxWarning: invalid escape sequence '\ '


Unziping /content/Roboflow Symbols Batch/Third Batch Z1.v9i.yolov8.zip to /content/combine_symbol_v0/dataset_03_01


...

<string>:1: SyntaxWarning: invalid escape sequence '\ '


Unziping /content/Roboflow Symbols Batch/Batch 7- raw_annotated_dataset_05.v8i.yolov8.zip to /content/combine_symbol_v0/dataset_07


<string>:1: SyntaxWarning: invalid escape sequence '\ '


Unziping /content/Roboflow Symbols Batch/batch_8_raw_annotated_dataset.v7i.yolov8.zip to /content/combine_symbol_v0/dataset_08


<string>:1: SyntaxWarning: invalid escape sequence '\ '


### Dataset Verification

In [10]:
###################################################
# REFERENCE: Functions defined in Dataset Utils
#
# script for verifying the dataset
# format of input data
#   - images (.jpg, .png ....)
#   - labels (yolo_format_annotations(cls, x_cen, y_cen, width, height)) # normalized values
#####################################################
import random

if __name__ == "__main__":
  colors = {}

  images_path = "/content/new_project_yolo/train/images"
  labels_path = "/content/new_project_yolo/train/labels"

  save_path = "/content/tests_saved_207"
  samples = 50 # number of samples to be used
  patch_size = 1792  # Not directly used unless you want to draw grid or check sizes

  os.makedirs(save_path, exist_ok=True)


  for img_idx, img_file in enumerate(random.choices(os.listdir(images_path), k=samples)):

    if not img_file.lower().endswith(('.jpg', '.png', '.jpeg')):
        continue

    image_path = os.path.join(images_path, img_file)
    label_path = os.path.join(labels_path, img_file.rsplit('.', 1)[0] + ".txt")

    image_save_path = os.path.join(save_path, img_file)

    verification_main(image_path, label_path, image_save_path)


    if img_idx >= samples:
      break

Image /content/new_project_yolo/train/images/6_0_0_img__jpg.rf.eG7HEs5ozlQ5qtxT7z5s.jpg Saved to /content/tests_saved_207/6_0_0_img__jpg.rf.eG7HEs5ozlQ5qtxT7z5s.jpg
Image /content/new_project_yolo/train/images/3_0_0_img__jpg.rf.T64ffzdYrAvcRQBHK0CN.jpg Saved to /content/tests_saved_207/3_0_0_img__jpg.rf.T64ffzdYrAvcRQBHK0CN.jpg
Image /content/new_project_yolo/train/images/8_0_0_img__jpg.rf.0r70Y0cYpXHcgQ4UHYMV.jpg Saved to /content/tests_saved_207/8_0_0_img__jpg.rf.0r70Y0cYpXHcgQ4UHYMV.jpg
Image /content/new_project_yolo/train/images/7_0_0_img__jpg.rf.nmrjNaw9qRPjAv9OXvP3.jpg Saved to /content/tests_saved_207/7_0_0_img__jpg.rf.nmrjNaw9qRPjAv9OXvP3.jpg
Image /content/new_project_yolo/train/images/1_0_0_img__jpg.rf.jpG2IgcUPpPWczioSlZR.jpg Saved to /content/tests_saved_207/1_0_0_img__jpg.rf.jpG2IgcUPpPWczioSlZR.jpg
Image /content/new_project_yolo/train/images/10_0_0_img__jpg.rf.GP34RnCGvqG0mA2TWRBb.jpg Saved to /content/tests_saved_207/10_0_0_img__jpg.rf.GP34RnCGvqG0mA2TWRBb.jpg
Image /c

In [6]:
!rm -rf /content/tests_saved_207

### Generalizing Class Of Sub Batches

In [8]:
if __name__=="__main__":

  # vars
  generalized_classes = {}


  # batches paths
  root_path = "/content/combine_symbol_v0"
  list_batches = os.listdir(root_path)


  # global dataset paths
  save_base_path = "/content/combine_symbol_v2"
  os.makedirs(save_base_path, exist_ok=True)


  for batch_idx, batch in enumerate(list_batches):
    print("Processing Batch {}/{}".format(batch_idx+1, len(list_batches)))

    # batch
    batch_path = os.path.join(root_path, batch)
    main(batch_path, save_base_path, g_idx=batch_idx)

Processing Batch 1/10
['0', '2 Way Angle Valve With Handle', '2 Way Valve', '2 Way Valve With Actuator', '3 Way Pressure Safety Valve', '3 Way Valve', '3-WAY SOLENOID VALVE', 'Actuator', 'Ball Valve', 'Ball Valve With Handle', 'Bearing', 'Bird Screen', 'Blind Flange', 'Break Symbol', 'Butterfly Valve', 'CP Type2', 'Cap', 'Check Valve', 'Computer Function', 'Computer Normally Accessible To Operator', 'Concentric', 'Cooler', 'Coupling', 'Cyclone Seperator', 'Cylinder', 'DCS', 'DCS Func Access in Aux Loc', 'DCS Func Field Mounted', 'DCS Func Inaccess in Prime Loc', 'DCS Single-Func Field Mounted', 'DCS Single-Func access in aux location', 'DCS Single-Func access in prime location', 'DIAPHRAGM VALVE', 'Double Block Valve', 'Drain', 'Eccentric', 'Electric Motor', 'Electromagnetic flowmeter', 'Emergency Stop', 'Equipment Filter', 'FILLER OR BREATHER', 'Filter', 'Fire Hydrant', 'Flange', 'Flanged Nozzle', 'Flanged Nozzle-Internal', 'Float Valve', 'Flow Direction', 'Flow Element', 'Flow Guage'

In [ ]:
!pip install -q roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 98.6 MB/s eta 0:00:00


loading Roboflow workspace...
loading Roboflow project...
Uploading to existing project institute-of-management-sciences-eacvm/symbol_dataset_fine_tune
[UPLOADED] /content/final_dataset/images/3.jpg (0purZZqFdaRtk5MFdmY8) [1.2s] / annotations = OK [0.4s]
[UPLOADED] /content/final_dataset/images/2.jpg (VXHoaLUTzrabkYU2nXtZ) [1.3s] / annotations = OK [0.4s]
[UPLOADED] /content/final_dataset/images/0.jpg (4F5M6t6EUbVblLxflgtD) [1.4s] / annotations = OK [0.4s]
[UPLOADED] /content/final_dataset/images/5.jpg (GG1V43YKkzwlud9PiHxS) [1.4s] / annotations = OK [0.5s]
[UPLOADED] /content/final_dataset/images/4.jpg (0purZZqFdaRtk5MFdmY8) [1.3s] / annotations = OK [0.7s]
[UPLOADED] /content/final_dataset/images/1.jpg (GG1V43YKkzwlud9PiHxS) [1.7s] / annotations = OK [0.6s]
[UPLOADED] /content/final_dataset/images/8.jpg (7H0oKJsQSbcGJd1yq14q) [0.8s] / annotations = OK [0.3s]
[UPLOADED] /content/final_dataset/images/7.jpg (jB0vu61n44TJLZcdqom7) [1.5s] / annotations = OK [1.3s]
[UPLOADED] /content/fina

### Classes Analytic & Jsonification

In [ ]:
# code for this in the utility levels

### Project Base Datasets

In [ ]:
# -- TIPS FOR SYNTHETIC DATA -- #

'''
1. Try to keep the distribution of *synthetic data* as much close as possible the original\
   distribution of the data.
2. Try to keep high proportion of the new symbols.
3. For more images keep it dense and for very some images keep it sparse.
4. Consider the original diagram and flip the symbols accordingly for the synthetic dataset.
5. Don't randomly orient/augment the symbols, give it more 1 hour to capture the real data\
   group them well and then give it proper augmentation (example the opc).
6. The Size the of the symbols is greater then the actual diagrams symbols size. \
   (this is not realy a problem for mdoel because model (rf-detr) sees patches not the whole image)
7. Try to keep the high proportion of those symbols which appears highly in the project. (IMPORTANT)
8. ** VERY IMPORTANT ** Consider the appearance of the symbols and it's placement if possible. (EXAMPLE IS OPC)



**** MEETING POINTS ****
1. Temperature Temperature stack
2. Temperature Pressure Stack
3. Liquid Flow Gas Flow
'''

#### Unzip All Projects Synthetic Datasets

In [4]:
# -- unzipping --
# synthetic data generated for the project
# 201, 202, 207

import os

path_01 = "/content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/201_properties.zip"
print("Unzipping {} to {}".format(path_01, path_01.split("/")[-1][:-4]))
!unzip -q {path_01} -d {path_01.split("/")[-1][:-4]}
print("Unzip file saved successfully to {}\n".format(path_01.split("/")[-1][:-4]))

path_02 = "/content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/new_valve.zip"
print("Unzipping {} to {}".format(path_02, path_02.split("/")[-1][:-4]))
!unzip -q {path_02} -d {path_02.split("/")[-1][:-4]}
print("Unzip file saved successfully to {}\n".format(path_02.split("/")[-1][:-4]))

path_03 = "/content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/opc_new.zip"
print("Unzipping {} to {}".format(path_03, path_03.split("/")[-1][:-4]))
!unzip -q {path_03} -d {path_03.split("/")[-1][:-4]}
print("Unzip file saved successfully to {}\n".format(path_03.split("/")[-1][:-4]))

Unzipping /content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/201_properties.zip to 201_properties
Unzip file saved successfully to 201_properties

Unzipping /content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/new_valve.zip to new_valve
Unzip file saved successfully to new_valve

Unzipping /content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/opc_new.zip to opc_new
Unzip file saved successfully to opc_new



In [7]:
# -- unzipping the updated opc and property data -- #


# -- updated new_opc --
path_ = "/content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/opc_new_updated.zip"
print("Unzipping {} to {}".format(path_, path_.split("/")[-1][:-4]))
!unzip -q {path_} -d {path_.split("/")[-1][:-4]}
print("Unzip file saved successfully to {}\n".format(path_.split("/")[-1][:-4]))


-- updated 201_properties --
path_ = "/content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/201_properties_updated.zip"
print("Unzipping {} to {}".format(path_, path_.split("/")[-1][:-4]))
!unzip -q {path_} -d {path_.split("/")[-1][:-4]}
print("Unzip file saved successfully to {}\n".format(path_.split("/")[-1][:-4]))


# -- explicit temp_pressure --
path_ = "/content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/temp_pressure.zip"
print("Unzipping {} to {}".format(path_, path_.split("/")[-1][:-4]))
!unzip -q {path_} -d {path_.split("/")[-1][:-4]}
print("Unzip file saved successfully to {}\n".format(path_.split("/")[-1][:-4]))


# -- updated gas_liquid_flow --
path_ = "/content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/gas_liquid_flow.zip"
print("Unzipping {} to {}".format(path_, path_.split("/")[-1][:-4]))
!unzip -q {path_} -d {path_.split("/")[-1][:-4]}
print("Unzip file saved successfully to {}\n".format(path_.split("/")[-1][:-4]))


# -- new_project shared via whatsapp (Tanzim) --
path_ = "/content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/new_project_yolo.zip"
print("Unzipping {} to {}".format(path_, path_.split("/")[-1][:-4]))
!unzip -q {path_} -d {path_.split("/")[-1][:-4]}
print("Unzip file saved successfully to {}\n".format(path_.split("/")[-1][:-4]))

Unzipping /content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/new_project_yolo.zip to new_project_yolo
Unzip file saved successfully to new_project_yolo



In [3]:
!rm -rf /content/201_properties

#### Copy Into Clean Format

In [8]:
import os

# dataset structure
root_dir = "/content/roboflow_upload/"


# -- copy 201 --
os.makedirs(os.path.join(root_dir, "project_201", "train", "images"), exist_ok=True)
os.makedirs(os.path.join(root_dir, "project_201", "train", "labels"), exist_ok=True)

!cp -r /content/201_properties/content/output/images/* -d /content/roboflow_upload/project_201/train/images
!cp -r /content/201_properties/content/output/labels/* -d /content/roboflow_upload/project_201/train/labels
print("PROJECT: Sythetic_201\nTotal Synthetic Train Images: {}"
      .format(len(os.listdir("/content/roboflow_upload/project_201/train/images"))))
print("\n")


# -- copy new_valve --
os.makedirs(os.path.join(root_dir, "project_new_valve", "train", "images"), exist_ok=True)
os.makedirs(os.path.join(root_dir, "project_new_valve", "train", "labels"), exist_ok=True)

!cp -r /content/new_valve/content/output/images/* -d /content/roboflow_upload/project_new_valve/train/images
!cp -r /content/new_valve/content/output/labels/* -d /content/roboflow_upload/project_new_valve/train/labels
print("PROJECT: Sythetic_new_valve\nTotal Synthetic Train Images: {}"
      .format(len(os.listdir("/content/roboflow_upload/project_new_valve/train/images"))))
print("\n")


# -- copy opc_new --
os.makedirs(os.path.join(root_dir, "project_opc_new", "train", "images"), exist_ok=True)
os.makedirs(os.path.join(root_dir, "project_opc_new", "train", "labels"), exist_ok=True)

!cp -r /content/opc_new/content/output/images/* -d /content/roboflow_upload/project_opc_new/train/images
!cp -r /content/opc_new/content/output/labels/* -d /content/roboflow_upload/project_opc_new/train/labels
print("PROJECT: Sythetic_opc_new\nTotal Synthetic Train Images: {}"
      .format(len(os.listdir("/content/roboflow_upload/project_opc_new/train/images"))))
print("\n")


# -- new_project shared via whatsapp (Tanzim) --
os.makedirs(os.path.join(root_dir, "new_project_yolo", "train", "images"), exist_ok=True)
os.makedirs(os.path.join(root_dir, "new_project_yolo", "train", "labels"), exist_ok=True)

!cp -r /content/new_project_yolo/train/images/* -d /content/roboflow_upload/new_project_yolo/train/images
!cp -r /content/new_project_yolo/train/labels/* -d /content/roboflow_upload/new_project_yolo/train/labels
print("PROJECT: new_project_yolo\nTotal Synthetic Train Images: {}"
      .format(len(os.listdir("/content/roboflow_upload/new_project_yolo/train/images"))))
print("\n")

cp: cannot stat '/content/201_properties/content/output/images/*': No such file or directory
cp: cannot stat '/content/201_properties/content/output/labels/*': No such file or directory
PROJECT: Sythetic_201
Total Synthetic Train Images: 0


cp: cannot stat '/content/new_valve/content/output/images/*': No such file or directory
cp: cannot stat '/content/new_valve/content/output/labels/*': No such file or directory
PROJECT: Sythetic_new_valve
Total Synthetic Train Images: 0


cp: cannot stat '/content/opc_new/content/output/images/*': No such file or directory
cp: cannot stat '/content/opc_new/content/output/labels/*': No such file or directory
PROJECT: Sythetic_opc_new
Total Synthetic Train Images: 0


PROJECT: new_project_yolo
Total Synthetic Train Images: 18




In [9]:
# dataset structure
root_dir = "/content/roboflow_upload/"


# -- 201_properties_updated --
os.makedirs(os.path.join(root_dir, "201_properties_updated", "train", "images"), exist_ok=True)
os.makedirs(os.path.join(root_dir, "201_properties_updated", "train", "labels"), exist_ok=True)

!cp -r /content/201_properties_updated/content/output/images/* -d /content/roboflow_upload/201_properties_updated/train/images
!cp -r /content/201_properties_updated/content/output/labels/* -d /content/roboflow_upload/201_properties_updated/train/labels
print("PROJECT: 201_properties_updated\nTotal Synthetic Train Images: {}"
      .format(len(os.listdir("/content/roboflow_upload/201_properties_updated/train/images"))))
print("\n")


# -- opc_new_updated --
os.makedirs(os.path.join(root_dir, "opc_new_updated", "train", "images"), exist_ok=True)
os.makedirs(os.path.join(root_dir, "opc_new_updated", "train", "labels"), exist_ok=True)

!cp -r /content/opc_new_updated/content/output/images/* -d /content/roboflow_upload/opc_new_updated/train/images
!cp -r /content/opc_new_updated/content/output/labels/* -d /content/roboflow_upload/opc_new_updated/train/labels
print("PROJECT: opc_new_updated\nTotal Synthetic Train Images: {}"
      .format(len(os.listdir("/content/roboflow_upload/opc_new_updated/train/images"))))
print("\n")


# -- temp_pressure --
os.makedirs(os.path.join(root_dir, "temp_pressure", "train", "images"), exist_ok=True)
os.makedirs(os.path.join(root_dir, "temp_pressure", "train", "labels"), exist_ok=True)

!cp -r /content/temp_pressure/content/output/images/* -d /content/roboflow_upload/temp_pressure/train/images
!cp -r /content/temp_pressure/content/output/labels/* -d /content/roboflow_upload/temp_pressure/train/labels
print("PROJECT: temp_pressure\nTotal Synthetic Train Images: {}"
      .format(len(os.listdir("/content/roboflow_upload/temp_pressure/train/images"))))
print("\n")


# -- gas_liquid_flow --
os.makedirs(os.path.join(root_dir, "gas_liquid_flow", "train", "images"), exist_ok=True)
os.makedirs(os.path.join(root_dir, "gas_liquid_flow", "train", "labels"), exist_ok=True)

!cp -r /content/gas_liquid_flow/content/output/images/* -d /content/roboflow_upload/gas_liquid_flow/train/images
!cp -r /content/gas_liquid_flow/content/output/labels/* -d /content/roboflow_upload/gas_liquid_flow/train/labels
print("PROJECT: gas_liquid_flow\nTotal Synthetic Train Images: {}"
      .format(len(os.listdir("/content/roboflow_upload/gas_liquid_flow/train/images"))))
print("\n")



PROJECT: Sythetic_201
Total Synthetic Train Images: 588


PROJECT: Sythetic_201
Total Synthetic Train Images: 544


PROJECT: Sythetic_201
Total Synthetic Train Images: 344


PROJECT: Sythetic_201
Total Synthetic Train Images: 344




In [5]:
!rm -rf /content/roboflow_upload

#### Upload To Roboflow

In [12]:
!pip install -q roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 99.7 MB/s eta 0:00:00


In [13]:
from roboflow import Roboflow


def upload_to_roboflow(keys_, dataset_path_, workspace_, project_id_):
  """Uploading dataset to roboflow
  """

  rf = Roboflow(api_key=keys_)

  workspace = rf.workspace(workspace_)

  workspace.upload_dataset(
      dataset_path=dataset_path_,
      project_name=project_id_,
      num_workers=10
  )

In [16]:
# -- dataset 001 --
import os
from google.colab import userdata

# -- vars --
keys = userdata.get("r_b_ibrah")
workspace = "institute-of-management-sciences-eacvm"
project_id = "symbol_detection_v2-dtl7r"


# --  upload single project to roboflow --
'''
upload only one single folder (project)
format:
  - project_1
      - images
      - labels
'''
upload_path = "/content/Symbol_Cached-1/train"
upload_to_roboflow(keys_=keys,
                  dataset_path_=upload_path,
                  workspace_=workspace,
                  project_id_=project_id)



# -- upload multiple projects to roboflow -- #
'''
for uploading multiple projects into roboflow
format:
  - root_path
    - project_1
        - images
        - labels
    - project_2
        - images
        - labels
    ...
    ...
'''
# root_path = "/content/roboflow_upload"
# for project in os.listdir(root_path):
#   print("Upload For {} starts to roboflow project {}".format(project, project_id))
#   upload_path = os.path.join(root_path, project, "train")

#   upload_to_roboflow(keys_=keys,
#                     dataset_path_=upload_path,
#                     workspace_=workspace,
#                     project_id_=project_id)
#   print("Project {} Upload Successfully Completed!!!\n\n".format(project))

loading Roboflow workspace...
loading Roboflow project...
Uploading to existing project institute-of-management-sciences-eacvm/symbol_detection_v2-dtl7r
[UPLOADED] /content/Symbol_Cached-1/train/images/0_0_0_img__jpg.rf.d395beeaecf3029dfa9843f4e7e34ae4.jpg (iJlq3D1lYWl7ApXxMgsM) [1.3s] / annotations = OK [0.4s]
[UPLOADED] /content/Symbol_Cached-1/train/images/12_0_0_img__jpg.rf.3e29bdf02d3012d42db5a8116385cd0f.jpg (q01QZYBWPP09RGhqg2it) [1.4s] / annotations = OK [0.4s]
[UPLOADED] /content/Symbol_Cached-1/train/images/0_0_0_img__jpg.rf.c4420a9b50a87a6044e18423cb4161ab.jpg (OUPJvGZHjljnXvOi01Ah) [1.3s] / annotations = OK [0.5s]
[UPLOADED] /content/Symbol_Cached-1/train/images/10_0_0_img__jpg.rf.d71d41559396f6b3e18d301001c7a31b.jpg (PnJYy8XDmJ8wXBLi8K41) [1.2s] / annotations = OK [0.7s]
[UPLOADED] /content/Symbol_Cached-1/train/images/11_0_0_img__jpg.rf.6f5ef8ce2de7ffd9513333678ccf09a0.jpg (ESK1IEO9goIeg9dP5NT6) [1.5s] / annotations = OK [0.5s]
[UPLOADED] /content/Symbol_Cached-1/train/im

'\nfor uploading multiple projects into roboflow \nformat: \n  - root_path\n    - project_1\n        - images \n        - labels \n    - project_2\n        - images \n        - labels \n    ...\n    ...\n'

## Model Training

### Dataset

In [2]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 120.1 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [3]:
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=str(userdata.get("r_b_ibrah")))
project = rf.workspace("institute-of-management-sciences-eacvm").project("symbol_detection_v2-dtl7r")
version = project.version(5)
dataset = version.download("coco")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to symbol_detection_v2-5 in coco:: 100%|██████████| 7184/7184 [00:02<00:00, 3068.37it/s]


### Model Archetecture

In [ ]:
!pip install wandb

In [1]:
import wandb
from google.colab import userdata

wandb.login(key=userdata.get("wandb"))
run = wandb.init(project="PNID_Models", entity="ibrahnengineer", job_type="download", anonymous="allow")

artifact = run.use_artifact("chines_model_v4:v0", type="model")
artifact_dir = artifact.download()


print("Artifact downloaded at:", artifact_dir)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ibrahnengineer to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


wandb: Downloading large artifact 'chines_model_v4:v0', 259.30MB. 2 files...
wandb:   2 of 2 files downloaded.  
Done. 00:00:09.1 (28.6MB/s)


Artifact downloaded at: /content/artifacts/chines_model_v4:v0


In [ ]:
!pip install -q rfdetr==1.6.0 supervision roboflow

In [ ]:
!pip install pytorch-lightning faster-coco-eval

In [ ]:
from rfdetr import RFDETRLarge

model = RFDETRLarge()

model.train(dataset_dir="/content/symbol_detection_v2-3",
            epochs=45, batch_size=6, grad_accum_steps=2,
            resume="/content/artifacts/chines_model:v0/24_march_detector.pth")

### Inference

In [4]:
import wandb
from google.colab import userdata

wandb.login(key=userdata.get("wandb"))
run = wandb.init(project="PNID_Models", entity="ibrahnengineer", job_type="download", anonymous="allow")

artifact = run.use_artifact("Project_Biased_Model:v0", type="model")
artifact_dir = artifact.download()


print("Artifact downloaded at:", artifact_dir)

invalid escape sequence '\/'
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ibrahnengineer to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


wandb: Downloading large artifact 'Project_Biased_Model:v0', 259.30MB. 2 files...
wandb:   2 of 2 files downloaded.  
Done. 00:00:11.7 (22.2MB/s)


Artifact downloaded at: /content/artifacts/Project_Biased_Model:v0


In [5]:
!pip install -q rfdetr==1.6.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 23.4 MB/s eta 0:00:00


In [6]:
from rfdetr import RFDETRLarge

model_path = "/content/artifacts/Project_Biased_Model:v0/checkpoint_best_total.pth"
model = RFDETRLarge(pretrain_weights=model_path)
model.optimize_for_inference()

[2026-04-09 00:56:42] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-04-09 00:56:42] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
`use_return_dict` is deprecated! Use `return_dict` instead!
Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a con

In [8]:
# project testing sets (diagrams, images)

!mkdir /content/projects_samples


!cp /content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/114-01-01-1100.jpg -d /content/projects_samples
!cp /content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/114-01-01-1101.jpg -d /content/projects_samples
!cp /content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/201_properties.jpg -d /content/projects_samples
!cp /content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/New_valve1_color_image.jpg -d /content/projects_samples
!cp /content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/New_valve_color_image.jpg -d /content/projects_samples
!cp /content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/OPC_New1_color_image_1.jpg -d /content/projects_samples
!cp /content/Ibrah/MyDrive/Symbol_Detection/Project_Base_Data/OPC_new_color_image_1.jpg -d /content/projects_samples

cp: cannot stat 'samples': No such file or directory


In [10]:
#===================================================#
#     SCRIPT WRITTEN BY GPT UPDATED BY IBRAH        #
#===================================================#

from PIL import Image
import numpy as np
import os
import cv2

def patch_image(image, img_name="img", patch_size=1792, save_dir="/content/patches", idx=0):
    """
    Splits image into patches of size patch_size x patch_size.
    Pads the last patches if needed.
    """

    os.makedirs(save_dir, exist_ok=True)

    if isinstance(image, Image.Image):
        image = np.array(image)

    h, w, c = image.shape
    patches = []

    for y in range(0, h, patch_size):
        for x in range(0, w, patch_size):

            patch = image[y:y+patch_size, x:x+patch_size]

            ph, pw = patch.shape[:2]

            # If patch is smaller → pad it
            if ph < patch_size or pw < patch_size:
                padded = np.zeros((patch_size, patch_size, c), dtype=image.dtype)
                padded[:ph, :pw] = patch
                patch = padded

            cv2.imwrite(os.path.join(save_dir, f"{img_name}_{idx}_{x}_{y}_img_.jpg"), patch)
            print("Patch {} Saved to {}".format(f"{img_name}_{idx}_{x}_{y}_img_.jpg", os.path.join(save_dir, f"{img_name}_{idx}_{x}_{y}_img_.jpg")))


if __name__=="__main__":

  # for single image

  # image_p = "/content/201.jpg"
  # image = Image.open(image_p)
  # patch_image(image, patch_size=1792, save_dir="/content/patches", idx=0)


  # for directory
  # if have multiple drawings. (unpatched diagrams/images)

  root_path = "/content/projects_samples"
  for idx, img_ in enumerate(os.listdir(root_path)):
    image_p = os.path.join(root_path, img_)
    image = Image.open(image_p)
    patch_image(image, img_name=img_, patch_size=1792, save_dir="/content/patches", idx=idx)


Patch KV-T0P1-PR-PD-1001-01.jpg_0_0_0_img_.jpg Saved to /content/patches/KV-T0P1-PR-PD-1001-01.jpg_0_0_0_img_.jpg
Patch KV-T0P1-PR-PD-1001-01.jpg_0_1792_0_img_.jpg Saved to /content/patches/KV-T0P1-PR-PD-1001-01.jpg_0_1792_0_img_.jpg
Patch KV-T0P1-PR-PD-1001-01.jpg_0_3584_0_img_.jpg Saved to /content/patches/KV-T0P1-PR-PD-1001-01.jpg_0_3584_0_img_.jpg
Patch KV-T0P1-PR-PD-1001-01.jpg_0_5376_0_img_.jpg Saved to /content/patches/KV-T0P1-PR-PD-1001-01.jpg_0_5376_0_img_.jpg
Patch KV-T0P1-PR-PD-1001-01.jpg_0_7168_0_img_.jpg Saved to /content/patches/KV-T0P1-PR-PD-1001-01.jpg_0_7168_0_img_.jpg
Patch KV-T0P1-PR-PD-1001-01.jpg_0_8960_0_img_.jpg Saved to /content/patches/KV-T0P1-PR-PD-1001-01.jpg_0_8960_0_img_.jpg
Patch KV-T0P1-PR-PD-1001-01.jpg_0_0_1792_img_.jpg Saved to /content/patches/KV-T0P1-PR-PD-1001-01.jpg_0_0_1792_img_.jpg
Patch KV-T0P1-PR-PD-1001-01.jpg_0_1792_1792_img_.jpg Saved to /content/patches/KV-T0P1-PR-PD-1001-01.jpg_0_1792_1792_img_.jpg
Patch KV-T0P1-PR-PD-1001-01.jpg_0_3584_1

In [11]:
#==============================#
#          IBRAH CODE          #
#==============================#

from PIL import Image
import random
import cv2
import numpy as np
import os


def draw(image, box, conf):

    # Random color for each box
    color = (random.randint(0,255), random.randint(0, 255), random.randint(0, 255))
    image = cv2.rectangle(image,
                          (int(box[0]), int(box[1])),
                          (int(box[2]), int(box[3])),
                          color, 2
                          )
    image = cv2.putText(image, f"{conf:.2f}",
                        (int(box[0]), int(box[1])),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        color,
                        1)
    return image



if __name__=="__main__":
  # paths
  image_dir = "/content/patches"
  save_dir = "/content/projects_tests_v4"
  os.makedirs(save_dir, exist_ok=True)


  for img_idx, img_name in enumerate(os.listdir(image_dir)):

    if not img_name.lower().endswith(".jpg"):
      continue

    # detection
    img_path = os.path.join(image_dir, img_name)
    image = Image.open(img_path)
    detections = model.predict(image, threshold=0.25)

    # xyxy, confs
    objs = detections.xyxy
    confs = detections.confidence
    image = np.array(image)

    # draw
    for Idx in range(len(objs)):
      image = draw(image, objs[Idx], confs[Idx])

    # write
    write_path = os.path.join(save_dir, f"{img_idx}_{img_name}")
    cv2.imwrite(write_path, image)
    print("No {} image {} saved to {}".format(img_idx, img_name, write_path))
    # break

No 0 image New_valve_color_image.jpg_2_7168_0_img_.jpg saved to /content/projects_tests_v4/0_New_valve_color_image.jpg_2_7168_0_img_.jpg
No 1 image 201_properties.jpg_6_1792_3584_img_.jpg saved to /content/projects_tests_v4/1_201_properties.jpg_6_1792_3584_img_.jpg
No 2 image KV-T0P1-PR-PD-1001-01.jpg_0_0_5376_img_.jpg saved to /content/projects_tests_v4/2_KV-T0P1-PR-PD-1001-01.jpg_0_0_5376_img_.jpg
No 3 image 207.jpg_1_3584_0_img_.jpg saved to /content/projects_tests_v4/3_207.jpg_1_3584_0_img_.jpg
No 4 image 201_properties.jpg_6_7168_5376_img_.jpg saved to /content/projects_tests_v4/4_201_properties.jpg_6_7168_5376_img_.jpg
No 5 image OPC_New1_color_image_1.jpg_3_1792_1792_img_.jpg saved to /content/projects_tests_v4/5_OPC_New1_color_image_1.jpg_3_1792_1792_img_.jpg
No 6 image OPC_New1_color_image_1.jpg_3_0_5376_img_.jpg saved to /content/projects_tests_v4/6_OPC_New1_color_image_1.jpg_3_0_5376_img_.jpg
No 7 image 201.jpg_5_1792_1792_img_.jpg saved to /content/projects_tests_v4/7_201.j

In [ ]:
!zip -r /content/validation_tests.zip /content/validation_results

In [12]:
!zip -r /content/projects_tests_v4.zip /content/projects_tests_v4

  adding: content/projects_tests_v4/ (stored 0%)
  adding: content/projects_tests_v4/87_New_valve1_color_image.jpg_9_3584_1792_img_.jpg (deflated 44%)
  adding: content/projects_tests_v4/106_KV-T0P1-PR-PD-1001-01.jpg_0_7168_3584_img_.jpg (deflated 34%)
  adding: content/projects_tests_v4/166_201.jpg_5_1792_0_img_.jpg (deflated 22%)
  adding: content/projects_tests_v4/30_211.jpg_8_3584_5376_img_.jpg (deflated 38%)
  adding: content/projects_tests_v4/55_114-01-01-1101.jpg_7_8960_3584_img_.jpg (deflated 27%)
  adding: content/projects_tests_v4/207_207.jpg_1_1792_3584_img_.jpg (deflated 63%)
  adding: content/projects_tests_v4/212_New_valve_color_image.jpg_2_5376_0_img_.jpg (deflated 22%)
  adding: content/projects_tests_v4/112_207.jpg_1_8960_3584_img_.jpg (deflated 98%)
  adding: content/projects_tests_v4/21_211.jpg_8_1792_0_img_.jpg (deflated 35%)
  adding: content/projects_tests_v4/225_OPC_new_color_image_1.jpg_10_5376_1792_img_.jpg (deflated 61%)
  adding: content/projects_tests_v4/156

In [13]:
!cp /content/projects_tests_v4.zip -d /content/Ibrah/MyDrive/Symbol_Detection/Transformer_Results

In [11]:
!cp /content/validation_tests.zip -d /content/Ibrah/MyDrive/Symbol_Detection/Transformer_Results

# General

In [ ]:
# curl to the dataset of symbol detection
!curl -L "https://app.roboflow.com/ds/D1PrFb2a2f?key=0K59Owr00J" > roboflow.zip; unzip roboflow.zip; rm roboflow.zip

In [ ]:
!curl -L "https://app.roboflow.com/ds/a2AX36g1NY?key=XdG0K22m0Q" > roboflow.zip;

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   925  100   925    0     0   2813      0 --:--:-- --:--:-- --:--:--  2811
100 3494M  100 3494M    0     0  31.9M      0  0:01:49  0:01:49 --:--:-- 32.3M


In [ ]:
!unzip /content/roboflow.zip -d /content/curl_dataset

Streaming output truncated to the last 5000 lines.
 extracting: /content/curl_dataset/train/bw_image_6_1_patch_1_4_jpg.rf.A53Q6DWZmOGnPWoAxV7L.jpg  
 extracting: /content/curl_dataset/train/bw_image_6_1_patch_1_6_jpg.rf.D5Hj8hsqhBvGU9R9MJOv.jpg  
 extracting: /content/curl_dataset/train/bw_image_6_1_patch_2_0_jpg.rf.sHlsg18SDeIWNjon2B2r.jpg  
 extracting: /content/curl_dataset/train/bw_image_6_1_patch_2508_5016_jpg.rf.sbetOUuxI6mmdri8q16B.jpg  
 extracting: /content/curl_dataset/train/bw_image_6_1_patch_1_2_jpg.rf.wIWLnB3Z8o5hMq9yQ6F5.jpg  
 extracting: /content/curl_dataset/train/bw_image_6_1_patch_1_7_jpg.rf.xq5ojNVdeKyhHpsF6EHz.jpg  
 extracting: /content/curl_dataset/train/bw_image_6_1_patch_1_6_jpg.rf.7f4pQDKD72ZwWyD25wyE.jpg  
 extracting: /content/curl_dataset/train/bw_image_6_1_patch_1_4_jpg.rf.Ehtetbv2I2yN4QyfEssq.jpg  
 extracting: /content/curl_dataset/train/bw_image_6_1_patch_1_5_jpg.rf.kJjp4ULMOrbNwnxI1ox9.jpg  
 extracting: /content/curl_dataset/train/bw_image_6_1_patch_1

In [ ]:
import os

len(os.listdir("/content/curl_dataset/train"))

6040

In [ ]:
from roboflow import Roboflow
from google.colab import userdata

rf = Roboflow(api_key=str(userdata.get("PNID_RB_API")))

workspace = rf.workspace("pnid-relgx")

workspace.upload_dataset(
    dataset_path="/content/curl_dataset/train",
    project_name="symbols_cached",
    num_workers=10
)

loading Roboflow workspace...
loading Roboflow project...


100%|██████████| 6039/6039 [00:00<00:00, 15395.68it/s]


Streaming output truncated to the last 5000 lines.
[UPLOADED] /content/curl_dataset/train/000220_jpg.rf.4LvzF83hQGXELioH7fou.jpg (h3Km8Zfg9YJs3dHW8GVa) [2.1s] / annotations = OK [0.4s]
[UPLOADED] /content/curl_dataset/train/000220_jpg.rf.WslDDNNP8vMiQuQRyp9y.jpg (5ZKrOiHWuA8cc7a82RIp) [2.0s] / annotations = OK [0.4s]
[UPLOADED] /content/curl_dataset/train/000220_jpg.rf.XxACcWHgYwKwKPGL6ARO.jpg (hWJUffO2Cn5XkWBrVfQN) [2.0s] / annotations = OK [0.4s]
[UPLOADED] /content/curl_dataset/train/000222_jpg.rf.iyLtHyPY61fIDQQbY8Ts.jpg (4EHMnHCwqLgpLB86Qy6Q) [1.0s] / annotations = OK [0.5s]
[UPLOADED] /content/curl_dataset/train/000222_jpg.rf.jKkd2slaKTTmscVtIRtN.jpg (PklYLKGRyJzZgqBZx2Jg) [1.0s] / annotations = OK [0.4s]
[UPLOADED] /content/curl_dataset/train/000220_jpg.rf.C7YQy1uU6yMyDicvAiI6.jpg (DomGZsmShvlE3UXh3dhu) [2.3s] / annotations = OK [0.5s]
[UPLOADED] /content/curl_dataset/train/000221_jpg.rf.XN2aUWYqVPqofHNOl2M4.jpg (AM3etkSzUvXXa5ygEESg) [2.0s] / annotations = OK [0.4s]
[UPLOADED] 